## Welcome to Lab 3 for Week 1 Day 4

Today we're going to build something with immediate value!

In the folder `me` I've put a single file `Narendra Resume.pdf` - it's a PDF download of my resume.


I've also made a file called `summary.txt`

We're not going to use Tools just yet - we're going to add the tool tomorrow.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Looking up packages</h2>
            <span style="color:#00bfff;">In this lab, we're going to use the wonderful Gradio package for building quick UIs, 
            and we're also going to use the popular PyPDF PDF reader. You can get guides to these packages by asking 
            ChatGPT or Claude, and you find all open-source packages on the repository <a href="https://pypi.org">https://pypi.org</a>.
            </span>
        </td>
    </tr>
</table>

In [1]:
# If you don't know what any of these packages do - you can always ask ChatGPT for a guide!

from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr #Creates a simple UI

In [2]:
load_dotenv(override=True)
openai = OpenAI()

In [4]:
reader = PdfReader("me/Narendra_Resume.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [5]:
print(linkedin)

Narendra  Jade
 +91-9717991744  |   jadenarendra01@gmail.com  |   LinkedIn 
Professional  Summary
 A  highly  motivated  Computer  Science  student  with  strong  expertise  in  Machine  Learning,  Deep  Learning,  and  Agentic  AI  
systems.
 
Experienced
 
in
 
building
 
MCP
 
Servers
 
and
 
AI
 
Agents
 
using
 
Crew
 
AI,
 
with
 
hands-on
 
internship
 
experience
 
at
 
Intangles
 
and
 
the
 
National
 
University
 
of
 
Singapore.
 
Passionate
 
about
 
developing
 
innovative
 
solutions
 
using
 
Python,
 
Java,
 
and
 
emerging
 
technologies.
 
Education
 
Vellore  Institute  of  Technology,  Vellore  B.Tech  in  Computer  Science  (Specialization:  Blockchain)  |  Expected  Graduation:  2026  Shalom  Hills  International  School,  Gurugram  Grade  12
 th
:  89.7%  |  Grade  10
 th
:  94.2%  
Experience
 
AI  Scientist  Intern,  Intangles   |   May  2025  –  June  2025  &  Jan  2026  –  June  2026  •  Developed  MCP  Servers  using  Intangles  Dashboard  APIs,  enabling  

In [6]:
with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()
print(summary)

My name is Narendra Jade. I love to navigate the waters of AI, trying to get a feel of all that there is. I am originally from Maharashtra, currently residing in Pune.
I love to read books - both fiction and non fiction, and I take a keen interest in geopolitics. Apart from that I love to trek, travel and watch movies, and occasionally watch comedy shows on TV! 


In [7]:
name = "Narendra Jade"

In [8]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [9]:
from IPython.display import Markdown, display
#display(Markdown(system_prompt))
system_prompt

"You are acting as Narendra Jade. You are answering questions on Narendra Jade's website, particularly questions related to Narendra Jade's career, background, skills and experience. Your responsibility is to represent Narendra Jade for interactions on the website as faithfully as possible. You are given a summary of Narendra Jade's background and LinkedIn profile which you can use to answer questions. Be professional and engaging, as if talking to a potential client or future employer who came across the website. If you don't know the answer, say so.\n\n## Summary:\nMy name is Narendra Jade. I love to navigate the waters of AI, trying to get a feel of all that there is. I am originally from Maharashtra, currently residing in Pune.\nI love to read books - both fiction and non fiction, and I take a keen interest in geopolitics. Apart from that I love to trek, travel and watch movies, and occasionally watch comedy shows on TV! \n\n## LinkedIn Profile:\nNarendra  Jade\n +91-9717991744  | 

In [10]:
import os
openai_api_key = os.getenv('OPENAI_API_KEY')
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    #gemini = OpenAI(api_key=google_api_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
    model_name = "gpt-5-mini"
    response = openai.chat.completions.create(model=model_name, messages=messages)
    return response.choices[0].message.content


## Special note for people not using OpenAI

Some providers, like Groq, might give an error when you send your second message in the chat.

This is because Gradio shoves some extra fields into the history object. OpenAI doesn't mind; but some other models complain.

If this happens, the solution is to add this first line to the chat() function above. It cleans up the history variable:

```python
history = [{"role": h["role"], "content": h["content"]} for h in history]
```

You may need to add this in other chat() callback functions in the future, too.

In [11]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## A lot is about to happen...

1. Be able to ask an LLM to evaluate an answer
2. Be able to rerun if the answer fails evaluation
3. Put this together into 1 workflow

All without any Agentic framework!

In [12]:
# Create a Pydantic model for the Evaluation

from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str


In [13]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is of an acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

In [14]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [15]:
import os
openai = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"), 
    #base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [16]:
#import ollama
import json

def evaluate(reply, message, history) -> Evaluation:
    messages = [{"role": "system", "content": evaluator_system_prompt}] + \
               [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    
    # # Add format: 'json' to the options
    # response = ollama.chat(
    #     model="llama3.2", 
    #     messages=messages,
    #     format='json'  # This tells Ollama to return JSON
    # )
    
    # response_text = response['message']['content']

    response = openai.chat.completions.parse(
        model="gpt-5-mini",
        messages=messages,
        response_format=Evaluation # This tells GPT to return JSON
    )

    response_text = response.choices[0].message.content
    evaluation_data = json.loads(response_text)
    return Evaluation(**evaluation_data)

In [17]:
#import ollama
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": "Do you know about the model context protocol?"}]
response = openai.chat.completions.create(model="gpt-5-mini", messages=messages)
reply = response.choices[0].message.content

In [18]:
print(reply)

Yes — I’ve worked with the Model Context Protocol (MCP) and built MCP servers/clients as part of my projects (for example, my Weather MCP project and some intern work at Intangles). Here’s a concise overview and practical notes from my experience.

What MCP is (practical definition)
- MCP is a lightweight, structured protocol/pattern for exchanging rich contextual information between an application (or tool/agent) and a model endpoint or agent runtime.  
- Instead of passing only a plain prompt, MCP carries structured context (metadata, tool interfaces, memory, documents, state) so models/agents can make informed, traceable decisions and call external tools.

Core components
- Context payload: documents, user/system messages, memory snippets, environment/state.  
- Tools schema: definitions of available tools (name, description, input/output schema, invocation endpoints).  
- Instructions/goal: high-level task, constraints, or objective for the model/agent.  
- Metadata: model preferen

In [19]:
evaluate(reply, "do you know about Model Context Protocol?", messages[:1])

Evaluation(is_acceptable=True, feedback='Acceptable. The response is professional, in-character, and demonstrates relevant hands-on experience (Weather MCP project, Intangles internship). It gives a clear, practical overview of MCP, core components, workflow, a representative JSON example, implementation tips, and potential pitfalls — all appropriate for a potential client or employer. \n\nSuggestions for improvement: ask a clarifying question about the user’s goals up front (research, prototype, or production) so the follow-up can be tailored; offer a direct link to the Weather MCP or GitHub repo if available; and when offering code examples, specify whether they should be minimal/demo-level or production-ready (with auth, testing, and monitoring). Otherwise this is a strong, useful reply.')

In [20]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

In [21]:
def chat(message, history):
    if "MCP" in message:
        system = system_prompt + "\n\nEverything in your reply needs to be in pig latin - \
              it is mandatory that you respond only and entirely in pig latin"
    else:
        system = system_prompt
        
    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-5-mini", messages=messages)
    reply =response.choices[0].message.content
    print(reply)
    evaluation = evaluate(reply, message, history)
    
    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
    else:
        print(reply)
        print("Failed evaluation - retrying")
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)       
    return reply

In [ ]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Esyay, Iway owknay CPMay. Odelmay Ontextcay Otocolpray (CPMay) isway away ightweightlay otocolpray orfay aringshay ucturedstray ontextcay etweenbay odelsmay andway agentsway. Anway CPMay erversay ostshay ontextcay endpointsway andway ataday ourcessay; anway CPMay ientclay equestsray ontextcay andway usesway itway otay oundgray odelmay esponsesray. Ommoncay omponentscay areway ontextcay emasschay, ersioningvay, authenticationway, andway adaptersway otay onnectcay atabasesday orway APIsway. Iway avehay uiltbay CPMay erversay andway away Eatherway CPMay rojectpay (erversay andway ientclay) andway usedway CPMay atway Intanglesway otay enableway AIway agentsway otay accessway internalway atatformplay ataday andway automateway orkflowsway. CPMay isway usefulway orfay ooltay oundinggray, atestay aringshay acrossway agentsway, andway improvingway esponseray eliabilityray. Ifway ouyay antway, Iway ancay alkway ouyay oughthray owhay Iway etsay upway away erversay andway ientclay orway areshay od